# Chapter 3: Building the Transformer block and a full GPT model

In the last chapter we built **multi-head attention**, the part of the model that lets every token look at the other tokens in a sentence. Attention is the heart of a GPT model, but on its own it is not a complete model. It needs a few supporting pieces around it to train well and to think more deeply about each token.

In this chapter we build those supporting pieces one at a time, snap them together into a **Transformer block**, and then stack many of those blocks into a complete **GPT model** that can produce text.

Here is the path we will follow:

- **Layer normalization**: keeps the numbers flowing through the model at a stable, healthy size
- **GELU activation**: a smooth "on/off switch" that lets the model learn non-linear patterns
- **Feed-forward network**: a small neural network that processes each token on its own
- **Residual connections**: shortcuts that help information and gradients travel through deep models
- **Transformer block**: attention + feed-forward + normalization + shortcuts, all in one reusable unit
- **GPT model**: embeddings, a stack of transformer blocks, and an output layer that scores every word in the vocabulary
- **Text generation**: a simple loop that makes the model write, one token at a time

A quick reminder about shapes, because we will check them constantly. Most tensors in this chapter have the shape `(batch, tokens, features)`:

- **batch**: how many sequences we process at the same time
- **tokens**: how many tokens are in each sequence
- **features**: how many numbers describe each token (also called the *embedding dimension*, `emb_dim`)

## Step 1: Layer normalization

As data passes through many layers of a deep network, the numbers inside it can drift. Some values grow very large, others shrink towards zero. When that happens, training becomes slow and unstable, because the model keeps chasing a moving target.

**Layer normalization** (LayerNorm) fixes this by rescaling the features of **each token** so that they have:

- a **mean of 0** (the values are centred around zero)
- a **variance of 1** (the values are spread out by a standard amount)

The formula is short. For the feature vector $x$ of one token:

$$\text{norm}(x) = \frac{x - \text{mean}(x)}{\sqrt{\text{var}(x) + \epsilon}}$$

What each part of the class below does:

- `mean` and `var` are computed over `dim=-1`, the **last dimension**, which holds the features of a single token. So every token is normalized on its own, independently of the other tokens and the other sequences in the batch.
- `keepdim=True` keeps the shape compatible, so the subtraction and division line up with `x` automatically.
- `unbiased=False` computes the plain variance (divide by `n`, not `n - 1`). This matches how GPT-2 was originally implemented.
- `eps` (epsilon) is a tiny number, `1e-5`, added before the square root so we never divide by zero.
- `scale` and `shift` are **trainable parameters**. Forcing every token to mean 0 and variance 1 can be too strict, so these let the model learn to stretch and move the values if that helps. They start as ones and zeros, which means "do nothing extra" at the beginning of training.

In [1]:
import torch 
import torch.nn as nn 
class LayerNormFromScratch(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps=eps
        self.scale=nn.Parameter(torch.ones(emb_dim))
        self.shift=nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean= x.mean(dim=-1, keepdim=True)
        var= x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x=(x-mean)/torch.sqrt(var+self.eps)
        return self.scale*norm_x+self.shift

## Checking that LayerNorm really works

Before trusting a layer, we test it. Here we create a small random tensor of shape `(2, 3, 5)`: **2** sequences, **3** tokens each, and **5** features per token.

After passing it through our LayerNorm, we expect two things:

1. **The shape does not change.** LayerNorm only rescales values, it never adds or removes any.
2. **Every token now has a mean close to 0 and a variance close to 1.**

When you look at the output, you may see means like `3.5e-08` instead of a perfect `0`. That is not a bug. It is `0.000000035`, a tiny rounding error that comes from how computers store decimal numbers. For all practical purposes, it is zero.

In [2]:
dummy_inputs= torch.randn(2,3,5)
ln_layer= LayerNormFromScratch(emb_dim=5)
ln_outputs= ln_layer(dummy_inputs)

print("input tensor shape:", dummy_inputs.shape)
print("output tensor shape:", ln_outputs.shape)
print("\n normalised output mean (should be close to 0):\n", ln_outputs.mean(dim=-1, keepdim=True))
print("\n normalised output var(should be close to 1):\n", ln_outputs.var(dim=-1, keepdim=True, unbiased=False))

input tensor shape: torch.Size([2, 3, 5])
output tensor shape: torch.Size([2, 3, 5])

 normalised output mean (should be close to 0):
 tensor([[[-2.9802e-08],
         [ 0.0000e+00],
         [-4.1723e-08]],

        [[-2.3842e-08],
         [ 0.0000e+00],
         [-2.2352e-08]]], grad_fn=<MeanBackward1>)

 normalised output var(should be close to 1):
 tensor([[[0.9998],
         [1.0000],
         [1.0000]],

        [[1.0000],
         [1.0000],
         [1.0000]]], grad_fn=<VarBackward0>)


## Step 2: The GELU activation function

A neural network made only of linear layers (multiply and add) can only learn straight-line relationships, no matter how many layers you stack. To learn the rich, curved patterns found in language, we need an **activation function** between the layers.

You may already know **ReLU**, which simply turns every negative number into 0 and keeps positive numbers as they are. It works, but it has a sharp corner at 0 and completely throws away all negative information.

GPT models use **GELU** (Gaussian Error Linear Unit) instead. You can think of it as a **smoother version of ReLU**:

- large positive values pass through almost unchanged
- large negative values are pushed close to 0
- values near 0 are handled with a smooth curve instead of a sharp corner, and small negative values are allowed to pass through a little

That smoothness makes the optimisation easier, which tends to help large models train better.

The exact GELU is expensive to compute, so GPT-2 used a fast and very accurate approximation based on `tanh`, which is exactly what this class implements:

$$\text{GELU}(x) \approx 0.5 \, x \left(1 + \tanh\left[\sqrt{2/\pi}\,\left(x + 0.044715\,x^3\right)\right]\right)$$

You do not need to memorise this formula. The key idea is simply: **GELU is a smooth on/off switch for each number.**

In [3]:
import torch 
import torch.nn as nn
class GELUActivation(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5*x*(1.0+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x+0.044715*torch.pow(x,3))))

## Step 3: The feed-forward network

Attention lets tokens **share information with each other**. The feed-forward network gives each token time to **think about that information on its own**.

It is a tiny neural network with three steps:

1. **Expand**: a linear layer grows each token's vector from `emb_dim` to `4 * emb_dim` features (for example, 256 becomes 1024).
2. **Activate**: GELU adds non-linearity, so the network can learn complex patterns.
3. **Shrink**: a second linear layer brings the vector back down to `emb_dim` features.

Why expand and then shrink? The larger middle layer gives the model a much bigger "workspace" to discover useful features. Shrinking back to the original size means the output fits perfectly into the next part of the model. The factor of 4 is the same choice used in the original Transformer and in GPT-2.

One important detail: the feed-forward network is applied to **every token separately**. Token 1 never sees token 2 inside this block. Mixing information between tokens is attention's job; refining each token is the feed-forward network's job.

`nn.Sequential` simply chains the three layers together, so the output of one becomes the input of the next.

In [4]:
import torch
import torch.nn as nn
class FeedForward(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.layers=nn.Sequential(nn.Linear(emb_dim, 4*emb_dim), GELUActivation(), nn.Linear(4*emb_dim, emb_dim))

    def forward(self,x):
        return self.layers(x)

## Checking the feed-forward shapes

We feed in a batch of shape `(8, 4, 256)`: **8** sequences, **4** tokens each, **256** features per token.

Inside the block, each token temporarily grows to 1024 features and then shrinks back. So from the outside, we expect the output to have **exactly the same shape as the input**: `(8, 4, 256)`.

"Same shape in, same shape out" is a property we want for every part of the transformer block, because it lets us stack the pieces on top of each other freely.

In [5]:
batch_inputs= torch.randn(8,4,256)
ffn_block=FeedForward(emb_dim=256)
ffn_outputs= ffn_block(batch_inputs)
print("input batch shape:", batch_inputs.shape)
print("ffn output shape:", ffn_outputs.shape)

input batch shape: torch.Size([8, 4, 256])
ffn output shape: torch.Size([8, 4, 256])


## Step 4: Bringing in multi-head attention from Chapter 2

The transformer block needs the `MultiHeadAttentionWrapper` we built in the previous chapter. Rather than copying and pasting that code here, we want to reuse it.

This cell installs `import-ipynb`, a small helper library that allows one Jupyter notebook to be imported like a regular Python module. The `%pip` command installs a package directly into the environment this notebook is running in.

In practice, the next cell uses a simpler approach (`%run`), so this installation is optional. It is kept here as an alternative way to share code between notebooks.

In [6]:
%pip install import-ipynb

Note: you may need to restart the kernel to use updated packages.


## Running the attention notebook

`%run` is a Jupyter "magic" command. It executes another notebook from top to bottom, **as if all of its cells were run right here**. After it finishes, everything that notebook defined becomes available in this notebook, including the two classes we need:

- `CausalAttention`
- `MultiHeadAttentionWrapper`

The path `../02_attention/attention.ipynb` means "go up one folder, then into the `02_attention` folder".

Because the whole attention notebook runs, you will also see all of its printed outputs appear below this cell. That is expected, and you can safely scroll past them.

In [7]:
%run ../02_attention/attention.ipynb

inputs shape: torch.Size([3, 3])
query: tensor([0.4400, 0.1500, 0.8900])
raw attention scores: tensor([1.0082, 0.9599, 0.4495])
attention weights(percentages): tensor([0.3961, 0.3774, 0.2265])
final context vector: tensor([0.5070, 0.5803, 0.6197])
all raw scores matrix:
 tensor([[0.9995, 0.9544, 0.4438],
        [0.9544, 1.4950, 1.1058],
        [0.4438, 1.1058, 1.0538]])

all attention weights matrix(row-wise sum=1):
 tensor([[0.3953, 0.3779, 0.2268],
        [0.2577, 0.4425, 0.2998],
        [0.2092, 0.4057, 0.3851]])

all final context vectors matrix:
 tensor([[0.5071, 0.5808, 0.6194],
        [0.5251, 0.6785, 0.5454],
        [0.5326, 0.7116, 0.4848]])
queries matrix shape: torch.Size([3, 2])
keys matrix shape: torch.Size([3, 2])
values matrix shape: torch.Size([3, 2])
scaled attention weights matrix:
 tensor([[0.3015, 0.4158, 0.2827],
        [0.3007, 0.4168, 0.2825],
        [0.3135, 0.3845, 0.3020]])

 final trainable context vector matrix shape: torch.Size([3, 2])
causal (GPT-s

## Step 5: Residual connections and the Transformer block

Now we assemble everything into a single **Transformer block**. Before looking at the code, there is one last idea to understand: **residual connections**.

### What is a residual connection?

A residual connection (also called a *shortcut* or *skip connection*) adds a layer's input back to its output:

```
output = x + layer(x)
```

Instead of asking each layer to produce a completely new representation, we only ask it to learn **what to add or change**. The original information always has a direct path forward.

This matters a lot in deep models. During training, the learning signal (the gradient) has to travel backwards through every layer. In a very deep network without shortcuts, that signal can shrink until it almost disappears, a problem known as **vanishing gradients**, and the early layers stop learning. The `x + ...` shortcut gives the gradient a clean highway back through the network.

### How the block works

The `forward` method has just two lines, and each follows the same pattern: **normalize, transform, drop out, add back**.

1. `x = x + self.dropout(self.att(self.ln1(x)))`
   - normalize the input with `ln1`
   - let the tokens exchange information through multi-head attention
   - apply dropout
   - add the result back to the original `x` (the residual connection)

2. `x = x + self.dropout(self.ff(self.ln2(x)))`
   - normalize again with `ln2`
   - let each token process its information through the feed-forward network
   - apply dropout
   - add the result back (another residual connection)

A few details worth knowing:

- **Two separate LayerNorms** (`ln1` and `ln2`) are used, because each one learns its own `scale` and `shift`.
- LayerNorm is applied **before** attention and the feed-forward network, not after. This is called **pre-LayerNorm**, and it is the arrangement GPT-2 uses because it makes training more stable.
- **Dropout** randomly switches off a small fraction of values during training. This stops the model from relying too heavily on any single path and helps it generalise. It is automatically disabled when the model is put into evaluation mode.
- `d_in` and `d_out` of the attention are both set to `emb_dim`, so the attention output has the same size as its input, which is exactly what the residual addition needs.

In [8]:
import torch
import torch.nn as nn 
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.att=MultiHeadAttentionWrapper(d_in=emb_dim, d_out=emb_dim, context_length=context_length, dropout=dropout, num_heads=num_heads, qkv_bias=qkv_bias)
        self.ff= FeedForward(emb_dim=emb_dim)
        self.ln1= LayerNormFromScratch(emb_dim=emb_dim)
        self.ln2= LayerNormFromScratch(emb_dim=emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        x=x+self.dropout(self.att(self.ln1(x)))
        x=x+self.dropout(self.ff(self.ln2(x)))

        return x 

## Testing the Transformer block

Time to check that all the pieces fit together. We pass a batch of shape `(8, 4, 256)` through one transformer block that uses **4 attention heads** and a dropout rate of `0.1`.

Internally a lot happens: normalization, attention across tokens, the feed-forward expansion to 1024 features and back, dropout, and two residual additions. But from the outside, we expect the output to have **exactly the same shape as the input**: `(8, 4, 256)`.

This is the most important property of a transformer block. Because the output looks just like the input, we can stack as many blocks as we want, one after another. That is precisely how real GPT models get their depth.

In [9]:
batch_inputs=torch.randn(8,4,256)
gpt_block= TransformerBlock(emb_dim=256, context_length=4, dropout=0.1, num_heads=4)
block_outputs= gpt_block(batch_inputs)
print("input batch shape:", batch_inputs.shape)
print("transformer block out shape:", block_outputs.shape)


input batch shape: torch.Size([8, 4, 256])
transformer block out shape: torch.Size([8, 4, 256])


## Step 6: Assembling the full GPT model

We now have every component we need. The `GPTMODEL` class connects them into a complete language model. Data flows through it in this order:

```
token IDs
   -> token embeddings + position embeddings
   -> dropout
   -> N transformer blocks
   -> final LayerNorm
   -> output layer
   -> logits (one score for every word in the vocabulary)
```

### The building blocks in `__init__`

- **`tok_emb` (token embedding)**: a lookup table that turns each token ID into a vector of `emb_dim` numbers. This is the same `nn.Embedding` idea from Chapter 1.
- **`pos_emb` (position embedding)**: a second lookup table, with one learnable vector for every position (0, 1, 2, ...). Attention on its own has no sense of word order; without this, *"dog bites man"* and *"man bites dog"* would look identical to the model. Adding a position vector tells each token **where** it sits in the sequence.
- **`drop_emb`**: dropout applied to the embeddings for regularisation.
- **`trf_blocks`**: `n_layers` transformer blocks chained together with `nn.Sequential`. The `*` unpacks the list, so each block is passed in as a separate layer. The output of one block becomes the input of the next.
- **`final_ln`**: one last LayerNorm to stabilise the values before the final prediction.
- **`out_head`**: a linear layer that maps each token's `emb_dim` vector to `vocab_size` numbers, one score for every possible next token. `bias=False` follows the GPT-2 design.

### What happens in `forward`

1. `in_idx` holds the token IDs, with shape `(batch, seq_len)`.
2. `tok_embeds` looks up a vector for every token, giving shape `(batch, seq_len, emb_dim)`.
3. `torch.arange(seq_len)` creates the positions `0, 1, 2, ..., seq_len - 1`, and `pos_embeds` looks up a vector for each position, giving shape `(seq_len, emb_dim)`. Passing `device=in_idx.device` makes sure the positions are created on the same device (CPU or GPU) as the input.
4. The token and position embeddings are **added together**. PyTorch automatically repeats the position embeddings for every sequence in the batch (this is called *broadcasting*).
5. The result goes through dropout, all the transformer blocks, and the final LayerNorm.
6. `out_head` produces the **logits** with shape `(batch, seq_len, vocab_size)`.

**Logits** are raw, unnormalised scores. A higher logit means the model thinks that token is a more likely next token. They are not probabilities yet, but applying softmax would turn them into probabilities.

In [10]:
class GPTMODEL(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb=nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb= nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb= nn.Dropout(cfg["drop_rate"])

        self.trf_blocks=nn.Sequential(*[TransformerBlock(emb_dim=cfg["emb_dim"], context_length=cfg["context_length"], 
        dropout=cfg["drop_rate"], num_heads=cfg["n_heads"], qkv_bias=cfg["qkv_bias"])
        for _ in range(cfg["n_layers"])])
        self.final_ln= LayerNormFromScratch(emb_dim=cfg["emb_dim"])
        self.out_head= nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self,in_idx):
        b,seq_len=in_idx.shape
        tok_embeds= self.tok_emb(in_idx)
        pos_embeds= self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x= self.drop_emb(tok_embeds+pos_embeds)
        x= self.trf_blocks(x)
        x=self.final_ln(x)
        logits= self.out_head(x)
        return logits 

## Configuring and testing a GPT-2 sized model

All the settings of the model live in a single dictionary, `GPT_CONFIG_124M`. Keeping them in one place makes it easy to build a smaller or larger model later by changing just a few numbers.

| Setting | Value | Meaning |
|---|---|---|
| `vocab_size` | 50257 | Number of tokens in the GPT-2 BPE tokenizer from Chapter 1 |
| `context_length` | 256 | Maximum number of tokens the model can look at in one go |
| `emb_dim` | 768 | Number of features used to describe each token |
| `n_heads` | 12 | Attention heads in each transformer block |
| `n_layers` | 12 | Number of transformer blocks stacked on top of each other |
| `drop_rate` | 0.1 | 10% dropout for regularisation |
| `qkv_bias` | False | Whether the query, key and value layers have a bias term |

These are the dimensions of the smallest GPT-2 model, often called "GPT-2 124M". The original GPT-2 used a context length of 1024; we use 256 to keep the model lighter while learning. Note that `qkv_bias` is listed for completeness, but the current model code does not pass it to the attention layers yet, so they use their default of no bias.

`torch.manual_seed(123)` fixes the random starting weights, so you get the same results every time you run this cell.

### The test

We pass in a single sequence of **3 token IDs**, `[[505, 707, 808]]`, so the input shape is `(1, 3)`: 1 sequence, 3 tokens.

The output shape should be `(1, 3, 50257)`. Reading it from left to right:

- **1** sequence in the batch
- **3** positions, one for each input token
- **50257** scores at every position, one for each word in the vocabulary

In other words, at **every** position the model predicts which token should come next. During text generation we will only care about the prediction at the last position.

In [11]:
GPT_CONFIG_124M= {"vocab_size": 50257, "context_length": 256, "emb_dim": 768, "n_heads": 12, "n_layers": 12, "drop_rate": 0.1, "qkv_bias": False}
torch.manual_seed(123)
model=GPTMODEL(GPT_CONFIG_124M)
sample_token_ids=torch.tensor([[505, 707, 808]])
output_logits = model(sample_token_ids)
print("input token batch shape:", sample_token_ids.shape)
print("final output logits shape:", output_logits.shape)

input token batch shape: torch.Size([1, 3])
final output logits shape: torch.Size([1, 3, 50257])


## Step 7: Counting parameters and generating text

### How big is our model?

The first line adds up the number of values in every trainable parameter tensor. `p.numel()` returns how many numbers a tensor holds, and the `requires_grad` check makes sure we only count weights that the model actually learns.

You will see about **162 million** parameters, even though the configuration is called "124M". The difference comes from the output layer. The original GPT-2 used a trick called **weight tying**: the output layer reuses the same weights as the token embedding layer, instead of having its own copy. Our model keeps them separate. The output layer alone holds `768 x 50257`, roughly **38.6 million** parameters, and removing it brings the count down to about **124 million**.

### Generating text, one token at a time

A GPT model never writes a full sentence in one step. It predicts **one next token**, adds it to the input, and repeats. `generate_text_simple` implements exactly this loop:

1. **Crop the context**: `idx[:, -context_size:]` keeps only the most recent tokens, because the model cannot look at more than `context_length` tokens at once.
2. **Run the model**: `torch.no_grad()` tells PyTorch we are not training, so it can skip tracking gradients. This saves memory and time.
3. **Keep only the last position**: `logits[:, -1, :]` selects the scores at the final position, because that is the model's guess for the token that comes **next**.
4. **Pick the best token**: `torch.argmax` chooses the token with the highest score. Always choosing the single most likely token is called **greedy decoding**.
5. **Append and repeat**: `torch.cat` adds the new token to the end of the sequence, and the loop runs again until `max_new_tokens` tokens have been added.

### Running it

We start with the token IDs `[40, 367, 2885]`, which the GPT-2 tokenizer decodes to *"I HAD"*, the opening words of *The Verdict* from Chapter 1. We ask for 5 new tokens, so the final output contains 3 + 5 = **8 token IDs**.

If you decode those new tokens back into text, they will be complete nonsense. **That is expected.** The model's weights are still random, because it has not been trained yet. The important result is that the full pipeline works from end to end: token IDs go in, and new token IDs come out.

Teaching the model to produce meaningful text is the job of **pretraining**, which is the next chapter.

In [12]:
total_params= sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total trainable parameter: {total_params:,}")

def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond= idx[:, -context_size:]

        with torch.no_grad():
            logits= model(idx_cond)
        logits = logits[:, -1, :]
        next_token= torch.argmax(logits, dim=-1, keepdim=True)
        idx=torch.cat((idx, next_token), dim=-1)
    return idx
start_context=torch.tensor([[40,367,2885]])

encoded_output= generate_text_simple(model, idx=start_context, max_new_tokens=5, context_size=GPT_CONFIG_124M["context_length"])

print("\nfinal encoded token ids output:", encoded_output)

total trainable parameter: 162,419,712

final encoded token ids output: tensor([[   40,   367,  2885, 40339, 33516, 20598, 38808, 43317]])


## Summary

In this chapter we turned a single attention layer into a complete GPT architecture:

- **LayerNorm** keeps every token's values at mean 0 and variance 1, so training stays stable.
- **GELU** is a smooth activation function that lets the model learn non-linear patterns.
- The **feed-forward network** expands each token to 4x its size, applies GELU, and shrinks it back, refining every token on its own.
- **Residual connections** add each sub-layer's input back to its output, giving information and gradients a shortcut through deep networks.
- The **Transformer block** combines all of the above with multi-head attention, and keeps the input and output shapes identical so blocks can be stacked.
- The **GPT model** adds token and position embeddings at the start, 12 stacked transformer blocks in the middle, and an output layer that scores every word in the vocabulary at the end.
- **Text generation** is a simple loop: predict the next token, append it, and repeat.

The architecture is finished, but the model still knows nothing, because its weights are random. In the next chapter we will **pretrain** it: define a loss function, write a training loop, and watch the generated text slowly turn from noise into real language.